# Customer Churn Prediction — EDA
**IBM Telco Kaggle Dataset | 7043 customers | 33 variables**

Download from: https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset

Save as `data/churn.csv` then run this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
os.makedirs('../plots', exist_ok=True)

df = pd.read_csv('../data/churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview & Missing Values

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nData types:')
print(df.dtypes)

## 2. Churn Distribution
Target column: **Churn Label** (Yes/No) or **Churn Value** (1/0)

In [ ]:
# Handle both column name variants
churn_col = 'Churn Label' if 'Churn Label' in df.columns else 'Churn'
counts = df[churn_col].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(counts.index, counts.values, color=['#2563EB','#DC2626'], alpha=0.85, edgecolor='white')
axes[0].set_title('Churn Count', fontsize=13, fontweight='bold')
for i,v in enumerate(counts.values): axes[0].text(i, v+30, str(v), ha='center', fontweight='bold')
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=['#2563EB','#DC2626'], startangle=90)
axes[1].set_title('Churn Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/eda_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Contract Type vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ct = df.groupby('Contract')[churn_col].value_counts(normalize=True).unstack() * 100
ct.plot(kind='bar', ax=ax, color=['#2563EB','#DC2626'], edgecolor='white', alpha=0.85)
ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
plt.tight_layout()
plt.savefig('../plots/eda_contract_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Tenure vs Churn

In [ ]:
tenure_col = 'Tenure Months' if 'Tenure Months' in df.columns else 'tenure'
fig, ax = plt.subplots(figsize=(8, 5))
for label, grp in df.groupby(churn_col):
    color = '#DC2626' if label == 'Yes' else '#2563EB'
    ax.hist(grp[tenure_col], bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
ax.set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout()
plt.savefig('../plots/eda_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Monthly Charges vs Churn

In [ ]:
charge_col = 'Monthly Charge' if 'Monthly Charge' in df.columns else 'MonthlyCharges'
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x=churn_col, y=charge_col, palette={'Yes':'#DC2626','No':'#2563EB'}, ax=ax)
ax.set_title('Monthly Charges vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/eda_monthly_charges.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Internet Service vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
internet = df.groupby('Internet Service')[churn_col].value_counts(normalize=True).unstack() * 100
internet.plot(kind='bar', ax=ax, color=['#2563EB','#DC2626'], edgecolor='white', alpha=0.85)
ax.set_title('Churn Rate by Internet Service', fontsize=13, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../plots/eda_internet_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Churn Reason Breakdown (unique to this dataset!)

In [ ]:
if 'Churn Reason' in df.columns:
    reasons = df[df['Churn Reason'].notna()]['Churn Reason'].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(9, 5))
    reasons.plot(kind='barh', ax=ax, color='#DC2626', alpha=0.8, edgecolor='white')
    ax.invert_yaxis()
    ax.set_title('Top 10 Churn Reasons', fontsize=13, fontweight='bold')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.savefig('../plots/eda_churn_reasons.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Churn Reason column not found.')

## Key EDA Findings

| Finding | Implication |
|---|---|
| ~27% churn rate | Imbalanced — use F1 and ROC-AUC, not accuracy |
| Month-to-Month → highest churn | Contract type is the strongest predictor |
| Low tenure → higher churn | Target early-stage customers for retention |
| Fiber Optic → higher churn | Price or service quality issue |
| Churn Reason available | Competitor / dissatisfaction split visible |

**Next:** `python src/train.py` to train all three models.